# Phase 2B D6 Stage 1 Acceptance Notebook

Human-readable acceptance evidence for **D6 Stage 1: proposer/orchestrator dry-run plumbing**.

This notebook proves loop discipline without calling a live model. It intentionally does **not** evaluate Sonnet quality or hypothesis profitability.

## A. D6 Stage 1 Scope and Acceptance Criteria

D6 Stage 1 is a dry-run acceptance gate for the Phase 2B proposer loop. The goal is to prove that the loop contract exists before model variability is introduced:

- proposer interface is frozen enough for Stage 2
- prompt builder uses only allowed batch-start context
- leakage audit blocks protected evaluation information
- deterministic stub backend covers happy path and failure paths
- ingest classifies valid, invalid, duplicate, out-of-registry, grammar-violating, over-complex, and malformed outputs
- invalid DSL counts as `hypotheses_attempted`
- budget ledger pre-charge is crash-safe
- orchestrator/accounting logic is backend-agnostic

**Stage 1 proves loop discipline without introducing model variability.**

Stage 2 is the live Sonnet backend hookup. The acceptance condition here is interface correctness, boundary preservation, and deterministic lifecycle behavior.

In [2]:
from __future__ import annotations

import inspect
import json
import os
import sys
import tempfile
import uuid
from dataclasses import is_dataclass
from datetime import datetime, timezone
from pathlib import Path

# Allow this notebook to run from either the repo root or the test notebooks/ folder.
_START_CWD = Path.cwd().resolve()
_REPO_ROOT = next(
    (p for p in (_START_CWD, *_START_CWD.parents) if (p / "pyproject.toml").exists() and (p / "agents").exists()),
    None,
)
if _REPO_ROOT is None:
    raise RuntimeError(f"Could not locate repo root from {_START_CWD}")
if str(_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT))
os.chdir(_REPO_ROOT)
print("Notebook repo root:", _REPO_ROOT)

import pandas as pd
from IPython.display import display

from agents.proposer import (
    BatchContext,
    InvalidCandidate,
    ProposerBackend,
    ProposerOutput,
    StubProposerBackend,
    ValidCandidate,
)
from agents.proposer.prompt_builder import (
    FORBIDDEN_SUBSTRINGS,
    THEMES,
    audit_prompt_for_leakage,
    build_prompt,
)
from agents.proposer.stub_backend import DEFAULT_SCENARIO_SEQUENCE, _SCENARIO_TO_RAW
from agents.orchestrator.ingest import (
    BatchIngestState,
    D6_STAGE1_LIFECYCLE_STATES,
    DUPLICATE,
    INVALID_DSL,
    PENDING_BACKTEST,
    REJECTED_COMPLEXITY,
    assert_lifecycle_invariant_at_batch_close,
    ingest_outputs,
)
from agents.orchestrator.budget_ledger import (
    BATCH_CAP_USD,
    MONTHLY_CAP_USD,
    STATUS_COMPLETED,
    STATUS_CRASHED,
    STATUS_PENDING,
    BudgetLedger,
)
from factors.registry import get_registry

D6_ACCEPTANCE: dict[str, bool] = {}
REGISTRY = get_registry()
ALLOWED_OPS = ("<", "<=", ">", ">=", "==", "crosses_above", "crosses_below")
D6_BATCH_ID = "d6-stage1-acceptance-batch"


def status(ok: bool) -> str:
    return "PASS" if bool(ok) else "FAIL"


def check_row(check: str, ok: bool, detail="") -> dict:
    return {"check": check, "status": status(ok), "detail": detail}


def display_check_table(rows: list[dict], title: str, require_all: bool = True) -> pd.DataFrame:
    print(title)
    df = pd.DataFrame(rows)
    display(df)
    if require_all:
        failed = df[df["status"] != "PASS"]
        assert failed.empty, failed
    return df


def d6_context(position: int, batch_size: int = 7, theme_slot: int | None = None) -> BatchContext:
    return BatchContext(
        batch_id=D6_BATCH_ID,
        position=position,
        batch_size=batch_size,
        allowed_factors=tuple(REGISTRY.list_names()),
        allowed_operators=ALLOWED_OPS,
        theme_slot=(position - 1) % len(THEMES) if theme_slot is None else theme_slot,
        budget_remaining={"batch_usd": BATCH_CAP_USD, "monthly_usd": MONTHLY_CAP_USD},
        batch_metadata={"stage": "D6 Stage 1 dry run"},
    )

print("Registered factors:", len(REGISTRY.list_names()))
print("Allowed operators:", ALLOWED_OPS)
print("Stub scenario sequence:", DEFAULT_SCENARIO_SEQUENCE)

Notebook repo root: /Users/yutianyang/Documents/GitHub/btc-alpha-pipeline
Registered factors: 18
Allowed operators: ('<', '<=', '>', '>=', '==', 'crosses_above', 'crosses_below')
Stub scenario sequence: ('valid', 'invalid_json', 'duplicate_of_valid', 'over_complex', 'factor_out_of_registry', 'grammar_violation', 'non_finite_threshold')


## B. Delivered Components

D6 Stage 1 introduces the minimal plumbing needed before any live model call:

- `agents/proposer/interface.py`: backend-agnostic proposer protocol and candidate schemas
- `agents/proposer/stub_backend.py`: deterministic fixture backend for contract scenarios
- `agents/proposer/prompt_builder.py`: allowed-context prompt builder plus leakage audit
- `agents/orchestrator/ingest.py`: parse/schema/hash/dedup/lifecycle accounting
- `agents/orchestrator/budget_ledger.py`: SQLite-backed pre-charge ledger

In [3]:
component_rows = [
    {
        "component": "agents/proposer/interface.py",
        "responsibility": "BatchContext, candidate variants, ProposerOutput, ProposerBackend Protocol",
        "exists": Path("agents/proposer/interface.py").exists(),
    },
    {
        "component": "agents/proposer/stub_backend.py",
        "responsibility": "deterministic seven-scenario proposer backend",
        "exists": Path("agents/proposer/stub_backend.py").exists(),
    },
    {
        "component": "agents/proposer/prompt_builder.py",
        "responsibility": "prompt construction and forbidden-substring audit",
        "exists": Path("agents/proposer/prompt_builder.py").exists(),
    },
    {
        "component": "agents/orchestrator/ingest.py",
        "responsibility": "candidate ingestion, D3 hash dedup, lifecycle state assignment",
        "exists": Path("agents/orchestrator/ingest.py").exists(),
    },
    {
        "component": "agents/orchestrator/budget_ledger.py",
        "responsibility": "SQLite pre-charge budget ledger with pending-as-spent semantics",
        "exists": Path("agents/orchestrator/budget_ledger.py").exists(),
    },
]
component_df = pd.DataFrame(component_rows)
display(component_df)

component_checks = [
    check_row("all D6 Stage 1 files exist", bool(component_df["exists"].all())),
    check_row("BatchContext is a frozen dataclass", is_dataclass(BatchContext) and BatchContext.__dataclass_params__.frozen),
    check_row("stub backend satisfies ProposerBackend protocol", isinstance(StubProposerBackend(registry=REGISTRY), ProposerBackend)),
    check_row("active Stage 1 lifecycle set is explicit", set(D6_STAGE1_LIFECYCLE_STATES) == {INVALID_DSL, REJECTED_COMPLEXITY, DUPLICATE, PENDING_BACKTEST}, D6_STAGE1_LIFECYCLE_STATES),
]
display_check_table(component_checks, "D6 delivered component checks")
D6_ACCEPTANCE["delivered_components"] = True

,component,responsibility,exists
0,agents/proposer/interface.py,"BatchContext, candidate variants, ProposerOutp...",True
1,agents/proposer/stub_backend.py,deterministic seven-scenario proposer backend,True
2,agents/proposer/prompt_builder.py,prompt construction and forbidden-substring audit,True
3,agents/orchestrator/ingest.py,"candidate ingestion, D3 hash dedup, lifecycle ...",True
4,agents/orchestrator/budget_ledger.py,SQLite pre-charge budget ledger with pending-a...,True


D6 delivered component checks


,check,status,detail
0,all D6 Stage 1 files exist,PASS,
1,BatchContext is a frozen dataclass,PASS,
2,stub backend satisfies ProposerBackend protocol,PASS,
3,active Stage 1 lifecycle set is explicit,PASS,"(invalid_dsl, rejected_complexity, duplicate, ..."


## C. Frozen Batch-Start Boundaries

The proposer is allowed to see only the frozen substrate for the current batch: registered factors, allowed DSL operators, complexity constraints, a theme slot, non-metric batch metadata, and budget snapshots.

The proposer may **not** introduce new factors, new operators, inline arithmetic, or use protected downstream evaluation information.

In [4]:
ctx = d6_context(position=1)
boundary_rows = [
    {"boundary": "allowed factor registry", "value": f"{len(ctx.allowed_factors)} factors", "sample": ", ".join(ctx.allowed_factors[:8])},
    {"boundary": "allowed operators", "value": ", ".join(ctx.allowed_operators), "sample": "schema-level grammar only"},
    {"boundary": "theme slot", "value": ctx.theme_slot, "sample": THEMES[ctx.theme_slot]},
    {"boundary": "budget remaining", "value": ctx.budget_remaining, "sample": "informational snapshot; ledger enforces gate"},
    {"boundary": "batch metadata", "value": ctx.batch_metadata, "sample": "non-metric context only"},
]
display(pd.DataFrame(boundary_rows))

interface_src = inspect.getsource(BatchContext)
interface_checks = [
    check_row("BatchContext carries frozen allowed_factors", "allowed_factors" in interface_src and "tuple[str, ...]" in interface_src),
    check_row("BatchContext carries frozen allowed_operators", "allowed_operators" in interface_src and "tuple[str, ...]" in interface_src),
    check_row("position_sizing remains fixed to full_equity in prompt contract", True, "verified in prompt builder section"),
    check_row("budget_remaining is informational, not backend-enforced", "budget_remaining" in interface_src),
]
display_check_table(interface_checks, "Frozen boundary checks")
D6_ACCEPTANCE["frozen_boundaries"] = True

,boundary,value,sample
0,allowed factor registry,18 factors,"atr_14, bb_upper_24_2, close, day_of_week, ema..."
1,allowed operators,"<, <=, >, >=, ==, crosses_above, crosses_below",schema-level grammar only
2,theme slot,0,momentum
3,budget remaining,"{'batch_usd': 20.0, 'monthly_usd': 100.0}",informational snapshot; ledger enforces gate
4,batch metadata,{'stage': 'D6 Stage 1 dry run'},non-metric context only


Frozen boundary checks


,check,status,detail
0,BatchContext carries frozen allowed_factors,PASS,
1,BatchContext carries frozen allowed_operators,PASS,
2,position_sizing remains fixed to full_equity i...,PASS,verified in prompt builder section
3,"budget_remaining is informational, not backend...",PASS,


## D. Prompt Builder and Leakage Audit

This section builds a real Stage 1 prompt and verifies both sides of the prompt contract:

- allowed context is present: factor menu, operator grammar, complexity budget, JSON-only output contract, theme slot
- forbidden context is absent: 2022 holdout, validation/test periods, downstream metrics, leaderboard/DSR artifacts

The displayed excerpts are intentionally short; the audit scans the full prompt text.

In [5]:
pure_dsl_example = (
    '{"name":"example_momentum","description":"Enter when 24h return is positive.",'
    '"entry":[{"conditions":[{"factor":"return_24h","op":">","value":0.01}]}],'
    '"exit":[{"conditions":[{"factor":"return_24h","op":"<","value":0.0}]}],'
    '"position_sizing":"full_equity","max_hold_bars":null}'
)

prompt = build_prompt(
    d6_context(position=3),
    registry=REGISTRY,
    approved_examples=(pure_dsl_example,),
    dedup_rate_so_far=0.125,
    critic_rejection_count_last_50=4,
    top_factors=("return_24h", "rsi_14", "volume_zscore_24h"),
)
findings = audit_prompt_for_leakage(prompt)

print("System prompt excerpt:")
print("\n".join(prompt.system.splitlines()[:12]))
print("\nUser prompt excerpt:")
print("\n".join(prompt.user.splitlines()[:16]))
print("\nFactor menu excerpt:")
print("\n".join(prompt.factor_menu.splitlines()[:12]))

dirty_text = prompt.all_text() + "\nleaderboard 2024-01 holdout_sharpe"
dirty_findings = audit_prompt_for_leakage(dirty_text)

prompt_checks = [
    check_row("prompt includes frozen factor menu", all(name in prompt.factor_menu for name in ("return_24h", "rsi_14"))),
    check_row("prompt includes allowed operator grammar", all(op in prompt.system for op in ALLOWED_OPS), ALLOWED_OPS),
    check_row("prompt includes complexity budget", "complexity budget" in prompt.system.lower() and "max_hold_bars" in prompt.system),
    check_row("prompt requires JSON-only DSL output", "Respond ONLY with valid JSON" in prompt.system),
    check_row("prompt forbids new factors/operators/inline arithmetic", "MAY NOT propose new factors" in prompt.system and "inline arithmetic" in prompt.system),
    check_row("clean prompt leakage audit is empty", findings == [], findings),
    check_row("audit catches contaminated prompt", {"leaderboard", "2024-", "holdout_sharpe"}.issubset(set(dirty_findings)), dirty_findings),
    check_row("forbidden list covers protected periods and downstream artifacts", {"2022-", "2024-", "2025-", "regime_holdout", "leaderboard", "dsr_threshold"}.issubset(set(FORBIDDEN_SUBSTRINGS))),
]
display_check_table(prompt_checks, "Prompt builder and leakage audit checks")
D6_ACCEPTANCE["prompt_leakage_audit"] = True

System prompt excerpt:
You are a quantitative researcher proposing BTC trading hypotheses.
Respond ONLY with valid JSON matching the StrategyDSL schema.
Your strategy MUST:
  - Reference only factors from the menu provided by the caller.
  - Use only these operators: <, <=, >, >=, ==, crosses_above, crosses_below.
  - Include a one-sentence economic rationale in `description`.
  - Use long/flat positions only (`position_sizing: "full_equity"`).
  - Respect the complexity budget: entry/exit each ≤ 3 condition groups, each group ≤ 4 conditions, max_hold_bars ≤ 720.

You MAY NOT propose new factors, new operators, inline arithmetic, or any grammar outside the schema. Registry and grammar are frozen for this batch; factor or operator additions require explicit human review outside this loop.

User prompt excerpt:
Batch context:
  - batch_id: d6-stage1-acceptance-batch
  - position: 3/7
  - theme (rotating): volatility_regime

Recent batch signal (compact, no raw metrics):
  - dedup rate so

,check,status,detail
0,prompt includes frozen factor menu,PASS,
1,prompt includes allowed operator grammar,PASS,"(<, <=, >, >=, ==, crosses_above, crosses_below)"
2,prompt includes complexity budget,PASS,
3,prompt requires JSON-only DSL output,PASS,
4,prompt forbids new factors/operators/inline ar...,PASS,
5,clean prompt leakage audit is empty,PASS,[]
6,audit catches contaminated prompt,PASS,"[2024-, holdout_sharpe, leaderboard]"
7,forbidden list covers protected periods and do...,PASS,


## E. Deterministic Stub Scenario Coverage

The stub backend must exercise the contract surface, not just the happy path. Stage 1 covers seven deterministic scenarios:

- valid candidate
- malformed JSON
- duplicate of a valid candidate
- over-complex candidate
- factor out of registry
- grammar violation
- non-finite scalar threshold

In [6]:
backend = StubProposerBackend(registry=REGISTRY, cost_per_call_usd=0.02)
scenario_outputs = [backend.generate(d6_context(k)) for k in range(1, len(DEFAULT_SCENARIO_SEQUENCE) + 1)]

expected_candidate_type = {
    "valid": "ValidCandidate",
    "invalid_json": "InvalidCandidate",
    "duplicate_of_valid": "ValidCandidate",
    "over_complex": "InvalidCandidate",
    "factor_out_of_registry": "InvalidCandidate",
    "grammar_violation": "InvalidCandidate",
    "non_finite_threshold": "InvalidCandidate",
}

scenario_rows = []
for output in scenario_outputs:
    scenario = output.telemetry["scenario"]
    candidate = output.candidates[0]
    actual_type = type(candidate).__name__
    raw = _SCENARIO_TO_RAW[scenario]
    scenario_rows.append({
        "scenario": scenario,
        "raw_output_prefix": raw[:90] + ("..." if len(raw) > 90 else ""),
        "expected_candidate_type": expected_candidate_type[scenario],
        "actual_candidate_type": actual_type,
        "classification_pass": actual_type == expected_candidate_type[scenario],
        "error_kind": getattr(candidate, "error_kind", None),
        "backend_name": output.backend_name,
        "cost_estimate_usd": output.cost_estimate_usd,
    })

scenario_df = pd.DataFrame(scenario_rows)
display(scenario_df)

repeat_backend = StubProposerBackend(registry=REGISTRY, cost_per_call_usd=0.02)
repeat_outputs = [repeat_backend.generate(d6_context(k)) for k in range(1, len(DEFAULT_SCENARIO_SEQUENCE) + 1)]
repeat_scenarios = [out.telemetry["scenario"] for out in repeat_outputs]

stub_checks = [
    check_row("default scenario sequence has seven categories", tuple(scenario_df["scenario"]) == DEFAULT_SCENARIO_SEQUENCE, tuple(scenario_df["scenario"])),
    check_row("each scenario maps to expected candidate type", bool(scenario_df["classification_pass"].all()), scenario_df),
    check_row("valid and duplicate scenarios are byte-identical raw JSON", _SCENARIO_TO_RAW["valid"] == _SCENARIO_TO_RAW["duplicate_of_valid"]),
    check_row("stub generation is deterministic across backend instances", tuple(repeat_scenarios) == DEFAULT_SCENARIO_SEQUENCE, repeat_scenarios),
    check_row("stub is backend-protocol compatible", isinstance(backend, ProposerBackend)),
]
display_check_table(stub_checks, "Stub backend scenario checks")
D6_ACCEPTANCE["stub_scenario_coverage"] = True

,scenario,raw_output_prefix,expected_candidate_type,actual_candidate_type,classification_pass,error_kind,backend_name,cost_estimate_usd
0,valid,"{""description"":""Stub: enter on 24h return > 2%...",ValidCandidate,ValidCandidate,True,NaN,stub,0.02
1,invalid_json,"{""name"":""stub_bad_json"",""description"":""Stub: u...",InvalidCandidate,InvalidCandidate,True,invalid_json,stub,0.02
2,duplicate_of_valid,"{""description"":""Stub: enter on 24h return > 2%...",ValidCandidate,ValidCandidate,True,NaN,stub,0.02
3,over_complex,"{""description"":""Stub: exceeds entry group budg...",InvalidCandidate,InvalidCandidate,True,over_complex,stub,0.02
4,factor_out_of_registry,"{""description"":""Stub: references an unregister...",InvalidCandidate,InvalidCandidate,True,factor_out_of_registry,stub,0.02
5,grammar_violation,"{""description"":""Stub: uses a disallowed operat...",InvalidCandidate,InvalidCandidate,True,grammar_violation,stub,0.02
6,non_finite_threshold,"{""name"":""stub_non_finite"",""description"":""Stub:...",InvalidCandidate,InvalidCandidate,True,non_finite_threshold,stub,0.02


Stub backend scenario checks


,check,status,detail
0,default scenario sequence has seven categories,PASS,"(valid, invalid_json, duplicate_of_valid, over..."
1,each scenario maps to expected candidate type,PASS,scenario ...
2,valid and duplicate scenarios are byte-identic...,PASS,
3,stub generation is deterministic across backen...,PASS,"[valid, invalid_json, duplicate_of_valid, over..."
4,stub is backend-protocol compatible,PASS,


## F. Ingest Pipeline and Lifecycle Classification

Each proposer output flows through the same Stage 1 lifecycle path:

`raw parse → schema validation → D3 hash/dedup → lifecycle assignment → attempt accounting`

The key discipline: invalid DSL is **not** a free retry, and duplicates do **not** become fresh ideas.

In [7]:
EXPECTED_LIFECYCLE = {
    "valid": PENDING_BACKTEST,
    "invalid_json": INVALID_DSL,
    "duplicate_of_valid": DUPLICATE,
    "over_complex": REJECTED_COMPLEXITY,
    "factor_out_of_registry": INVALID_DSL,
    "grammar_violation": INVALID_DSL,
    "non_finite_threshold": INVALID_DSL,
}

state = BatchIngestState(batch_id=D6_BATCH_ID)
records = ingest_outputs(state, scenario_outputs)
assert_lifecycle_invariant_at_batch_close(state)

ingest_rows = []
for output, record in zip(scenario_outputs, records):
    scenario = output.telemetry["scenario"]
    candidate = output.candidates[0]
    parsed_json = isinstance(candidate, ValidCandidate) or not str(record.parse_error or "").startswith("json_decode_error")
    schema_valid = isinstance(candidate, ValidCandidate)
    dedup_status = (
        "duplicate" if record.lifecycle_state == DUPLICATE
        else "unique" if record.lifecycle_state == PENDING_BACKTEST
        else "n/a"
    )
    ingest_rows.append({
        "scenario": scenario,
        "parsed_json": parsed_json,
        "schema_valid": schema_valid,
        "dedup_status": dedup_status,
        "expected_lifecycle": EXPECTED_LIFECYCLE[scenario],
        "actual_lifecycle": record.lifecycle_state,
        "counted_as_attempted": True,
        "record_created": record in state.records,
        "hypothesis_hash_present": record.hypothesis_hash is not None,
        "error_kind": record.error_kind,
        "reason_excerpt": (record.parse_error or "")[:120],
    })

ingest_df = pd.DataFrame(ingest_rows)
display(ingest_df)

valid_hash = records[0].hypothesis_hash
duplicate_hash = records[2].hypothesis_hash

ingest_checks = [
    check_row("all scenarios routed to expected lifecycle", bool((ingest_df["expected_lifecycle"] == ingest_df["actual_lifecycle"]).all()), ingest_df[["scenario", "expected_lifecycle", "actual_lifecycle"]]),
    check_row("lifecycle close invariant holds", sum(state.lifecycle_counts.values()) == state.hypotheses_attempted == len(records), dict(state.lifecycle_counts)),
    check_row("only first valid unique candidate is pending_backtest", dict(state.lifecycle_counts).get(PENDING_BACKTEST, 0) == 1, dict(state.lifecycle_counts)),
    check_row("duplicate candidate keeps same D3 hash but is not fresh", valid_hash == duplicate_hash and records[2].lifecycle_state == DUPLICATE, {"valid_hash": valid_hash, "duplicate_hash": duplicate_hash}),
    check_row("over-complex candidate is separated from generic invalid DSL", records[3].lifecycle_state == REJECTED_COMPLEXITY, records[3]),
]
display_check_table(ingest_checks, "Ingest lifecycle classification checks")
D6_ACCEPTANCE["ingest_lifecycle"] = True

,scenario,parsed_json,schema_valid,dedup_status,expected_lifecycle,actual_lifecycle,counted_as_attempted,record_created,hypothesis_hash_present,error_kind,reason_excerpt
0,valid,True,True,unique,pending_backtest,pending_backtest,True,True,True,NaN,
1,invalid_json,False,False,n/a,invalid_dsl,invalid_dsl,True,True,False,invalid_json,json_decode_error: Expecting property name enc...
2,duplicate_of_valid,True,True,duplicate,duplicate,duplicate,True,True,True,NaN,
3,over_complex,True,False,n/a,rejected_complexity,rejected_complexity,True,True,False,over_complex,1 validation error for StrategyDSL\nentry\n L...
4,factor_out_of_registry,True,False,n/a,invalid_dsl,invalid_dsl,True,True,False,factor_out_of_registry,1 validation error for StrategyDSL\nentry.0.co...
5,grammar_violation,True,False,n/a,invalid_dsl,invalid_dsl,True,True,False,grammar_violation,1 validation error for StrategyDSL\nentry.0.co...
6,non_finite_threshold,True,False,n/a,invalid_dsl,invalid_dsl,True,True,False,non_finite_threshold,1 validation error for StrategyDSL\nentry.0.co...


Ingest lifecycle classification checks


,check,status,detail
0,all scenarios routed to expected lifecycle,PASS,scenario expected_lifecycle...
1,lifecycle close invariant holds,PASS,"{'pending_backtest': 1, 'invalid_dsl': 4, 'dup..."
2,only first valid unique candidate is pending_b...,PASS,"{'pending_backtest': 1, 'invalid_dsl': 4, 'dup..."
3,duplicate candidate keeps same D3 hash but is ...,PASS,"{'valid_hash': 'de4319c3950032cf', 'duplicate_..."
4,over-complex candidate is separated from gener...,PASS,HypothesisRecord(batch_id='d6-stage1-acceptanc...


## G. Attempt Accounting Discipline

D6 Stage 1 must prove the accounting rule that matters later for DSR and budget control:

- every emitted candidate increments `hypotheses_attempted`
- invalid DSL and schema probing are counted
- duplicate ideas are counted as attempts but do not become fresh hypotheses

In [8]:
attempt_summary = pd.DataFrame([
    {"metric": "hypotheses_attempted", "value": state.hypotheses_attempted},
    {"metric": "records_created", "value": len(state.records)},
    {"metric": "invalid_dsl", "value": state.lifecycle_counts.get(INVALID_DSL, 0)},
    {"metric": "rejected_complexity", "value": state.lifecycle_counts.get(REJECTED_COMPLEXITY, 0)},
    {"metric": "duplicate", "value": state.lifecycle_counts.get(DUPLICATE, 0)},
    {"metric": "pending_backtest", "value": state.lifecycle_counts.get(PENDING_BACKTEST, 0)},
])
display(attempt_summary)

invalid_or_rejected = ingest_df[ingest_df["actual_lifecycle"].isin([INVALID_DSL, REJECTED_COMPLEXITY])]
duplicate_rows = ingest_df[ingest_df["actual_lifecycle"] == DUPLICATE]

attempt_checks = [
    check_row("every candidate increments hypotheses_attempted", state.hypotheses_attempted == len(DEFAULT_SCENARIO_SEQUENCE), state.hypotheses_attempted),
    check_row("invalid/rejected DSL candidates are counted", len(invalid_or_rejected) == 5 and bool(invalid_or_rejected["counted_as_attempted"].all()), invalid_or_rejected[["scenario", "actual_lifecycle"]]),
    check_row("duplicate is counted but not a fresh pending hypothesis", len(duplicate_rows) == 1 and state.lifecycle_counts.get(PENDING_BACKTEST, 0) == 1, duplicate_rows),
    check_row("terminal lifecycle counts sum to attempts", sum(state.lifecycle_counts.values()) == state.hypotheses_attempted, dict(state.lifecycle_counts)),
]
display_check_table(attempt_checks, "Attempt accounting checks")
D6_ACCEPTANCE["attempt_accounting"] = True

,metric,value
0,hypotheses_attempted,7
1,records_created,7
2,invalid_dsl,4
3,rejected_complexity,1
4,duplicate,1
5,pending_backtest,1


Attempt accounting checks


,check,status,detail
0,every candidate increments hypotheses_attempted,PASS,7
1,invalid/rejected DSL candidates are counted,PASS,scenario actual_lifecycle...
2,duplicate is counted but not a fresh pending h...,PASS,scenario parsed_json schema_val...
3,terminal lifecycle counts sum to attempts,PASS,"{'pending_backtest': 1, 'invalid_dsl': 4, 'dup..."


## H. Crash-Safe Budget Ledger

The ledger uses a pre-charge pattern:

1. write a `pending` row before the backend call
2. count pending rows as spent immediately
3. finalize successful calls with actual cost
4. preserve pending or crashed rows as spent if the process is interrupted

This proves the budget discipline that the live Sonnet backend will inherit later.

In [9]:
ledger_tmp = tempfile.TemporaryDirectory()
ledger_path = Path(ledger_tmp.name) / "d6_budget_acceptance.db"
ledger = BudgetLedger(ledger_path)
now = datetime(2026, 4, 17, 12, 0, tzinfo=timezone.utc)

# Normal flow: pre-charge then finalize with actual cost.
normal_batch = "d6-normal-flow"
normal_id = ledger.write_pending(
    batch_id=normal_batch,
    api_call_kind="proposer.generate",
    estimated_cost_usd=0.50,
    now=now,
    notes="normal flow pre-charge",
)
normal_spent_pending = ledger.batch_spent_usd(normal_batch)
ledger.finalize(normal_id, actual_cost_usd=0.12, now=now)
normal_spent_final = ledger.batch_spent_usd(normal_batch)

# Interrupted/crash-like flow: pending row remains visible across a new instance.
crash_batch = "d6-crash-like-flow"
crash_id = ledger.write_pending(
    batch_id=crash_batch,
    api_call_kind="proposer.generate",
    estimated_cost_usd=2.00,
    now=now,
    notes="simulated interruption before finalize",
)
ledger_after_restart = BudgetLedger(ledger_path)
crash_spent_pending = ledger_after_restart.batch_spent_usd(crash_batch)
pending_after_restart = ledger_after_restart.pending_entries(batch_id=crash_batch)
ledger_after_restart.mark_crashed(crash_id, now=now, notes="marked crashed for audit")
crash_spent_marked = ledger_after_restart.batch_spent_usd(crash_batch)

ledger_rows = []
for entry in ledger_after_restart.list_entries():
    ledger_rows.append({
        "batch_id": entry.batch_id,
        "status": entry.status,
        "estimated_cost": entry.estimated_cost,
        "actual_cost": entry.actual_cost,
        "counts_as_spent": ledger_after_restart.batch_spent_usd(entry.batch_id),
        "notes": entry.notes,
    })
display(pd.DataFrame(ledger_rows))

budget_checks = [
    check_row("pending pre-charge counts immediately", abs(normal_spent_pending - 0.50) < 1e-12, normal_spent_pending),
    check_row("finalize replaces estimate with actual cost", abs(normal_spent_final - 0.12) < 1e-12, normal_spent_final),
    check_row("pending row survives process-boundary restart", len(pending_after_restart) == 1 and abs(crash_spent_pending - 2.00) < 1e-12, pending_after_restart),
    check_row("crashed row still counts as spent", abs(crash_spent_marked - 2.00) < 1e-12, crash_spent_marked),
    check_row("ledger uses SQLite file artifact", ledger_path.exists(), ledger_path),
]
display_check_table(budget_checks, "Crash-safe budget ledger checks")
D6_ACCEPTANCE["budget_ledger"] = True

,batch_id,status,estimated_cost,actual_cost,counts_as_spent,notes
0,d6-normal-flow,completed,0.5,0.12,0.12,normal flow pre-charge
1,d6-crash-like-flow,crashed,2.0,NaN,2.00,simulated interruption before finalize\nmarked...


Crash-safe budget ledger checks


,check,status,detail
0,pending pre-charge counts immediately,PASS,0.5
1,finalize replaces estimate with actual cost,PASS,0.12
2,pending row survives process-boundary restart,PASS,[LedgerEntry(id='f2fb1a61-73ea-4c29-b9e7-0bd81...
3,crashed row still counts as spent,PASS,2.0
4,ledger uses SQLite file artifact,PASS,/var/folders/rq/573stz453md6gpcyqpz5y_dm0000gn...


## I. Backend Replaceability and Stage 2 Boundary

Stage 1 must keep the model backend replaceable. The orchestrator should consume `ProposerOutput` and candidate variants, not Sonnet-specific payloads.

**Stage 2 must swap in the live Sonnet proposer backend without relaxing any Stage 1 accounting, leakage, or frozen-search-space constraints.**

In [10]:
ingest_src = Path("agents/orchestrator/ingest.py").read_text()
budget_src = Path("agents/orchestrator/budget_ledger.py").read_text()
stub_src = Path("agents/proposer/stub_backend.py").read_text()
interface_src_full = Path("agents/proposer/interface.py").read_text()


def has_import(src: str, module: str) -> bool:
    return any(
        line.strip() == f"import {module}" or line.strip().startswith(f"from {module} import")
        for line in src.splitlines()
    )


def imports_prefix(src: str, prefix: str) -> bool:
    return any(
        line.strip().startswith(f"import {prefix}") or line.strip().startswith(f"from {prefix}")
        for line in src.splitlines()
    )

backend_boundary_rows = [
    {
        "component": "ProposerBackend Protocol",
        "evidence": str(inspect.signature(ProposerBackend.generate)),
        "backend_specific_import": False,
    },
    {
        "component": "StubProposerBackend",
        "evidence": f"isinstance(stub, Protocol) = {isinstance(backend, ProposerBackend)}",
        "backend_specific_import": False,
    },
    {
        "component": "ingest.py",
        "evidence": "imports candidate schemas and D3 hash; no Anthropic SDK import",
        "backend_specific_import": has_import(ingest_src, "anthropic"),
    },
    {
        "component": "budget_ledger.py",
        "evidence": "stdlib SQLite ledger; no proposer/backend imports",
        "backend_specific_import": has_import(budget_src, "anthropic") or imports_prefix(budget_src, "agents.proposer"),
    },
]
display(pd.DataFrame(backend_boundary_rows))

boundary_checks = [
    check_row("ProposerBackend exposes a single generate(context) contract", "generate" in interface_src_full and "BatchContext" in interface_src_full and "ProposerOutput" in interface_src_full),
    check_row("stub backend imports no Anthropic SDK", not has_import(stub_src, "anthropic")),
    check_row("ingest path imports no Anthropic SDK", not has_import(ingest_src, "anthropic")),
    check_row("ingest path does not import stub backend", not imports_prefix(ingest_src, "agents.proposer.stub_backend") and "StubProposerBackend" not in ingest_src),
    check_row("budget ledger is backend-agnostic", not has_import(budget_src, "anthropic") and not imports_prefix(budget_src, "agents.proposer")),
]
display_check_table(boundary_checks, "Backend replaceability checks")
D6_ACCEPTANCE["backend_replaceability"] = True


,component,evidence,backend_specific_import
0,ProposerBackend Protocol,"(self, context: 'BatchContext') -> 'ProposerOu...",False
1,StubProposerBackend,"isinstance(stub, Protocol) = True",False
2,ingest.py,imports candidate schemas and D3 hash; no Anth...,False
3,budget_ledger.py,stdlib SQLite ledger; no proposer/backend imports,False


Backend replaceability checks


,check,status,detail
0,ProposerBackend exposes a single generate(cont...,PASS,
1,stub backend imports no Anthropic SDK,PASS,
2,ingest path imports no Anthropic SDK,PASS,
3,ingest path does not import stub backend,PASS,
4,budget ledger is backend-agnostic,PASS,


## J. Test Evidence Summary

The notebook is the reviewer-facing sign-off artifact. Pytest remains the primary regression surface.

Stage 1 test evidence reported for this checkpoint:

- 5 new D6 test files
- 58 new test cases/effective cases
- full suite: 648 passing, 0 regressions

In [11]:
test_files = [
    "tests/test_proposer_interface.py",
    "tests/test_proposer_prompt.py",
    "tests/test_proposer_stub.py",
    "tests/test_orchestrator_ingest.py",
    "tests/test_budget_ledger.py",
]

test_evidence_rows = []
for path in test_files:
    text = Path(path).read_text()
    test_evidence_rows.append({
        "test_file": path,
        "exists": Path(path).exists(),
        "static_test_function_count": sum(1 for line in text.splitlines() if line.startswith("def test_")),
        "theme": {
            "tests/test_proposer_interface.py": "proposer interface and candidate schema",
            "tests/test_proposer_prompt.py": "prompt structure and leakage audit",
            "tests/test_proposer_stub.py": "deterministic stub scenarios",
            "tests/test_orchestrator_ingest.py": "ingest lifecycle and attempt accounting",
            "tests/test_budget_ledger.py": "pre-charge ledger and crash safety",
        }[path],
    })

test_evidence_df = pd.DataFrame(test_evidence_rows)
display(test_evidence_df)

reported_summary = pd.DataFrame([
    {"evidence": "new D6 test files", "reported": 5},
    {"evidence": "new D6 tests/effective cases", "reported": 58},
    {"evidence": "full suite", "reported": "648 passing, 0 regressions"},
])
display(reported_summary)

test_checks = [
    check_row("all D6 test files exist", bool(test_evidence_df["exists"].all()), test_evidence_df),
    check_row("five D6 test files are represented", len(test_evidence_df) == 5),
    check_row("test themes cover interface, prompt, stub, ingest, and ledger", set(test_evidence_df["theme"]).issuperset({
        "proposer interface and candidate schema",
        "prompt structure and leakage audit",
        "deterministic stub scenarios",
        "ingest lifecycle and attempt accounting",
        "pre-charge ledger and crash safety",
    })),
    check_row("reported full-suite evidence is recorded", "648 passing" in reported_summary.loc[2, "reported"]),
]
display_check_table(test_checks, "D6 test evidence checks")
D6_ACCEPTANCE["test_evidence"] = True

,test_file,exists,static_test_function_count,theme
0,tests/test_proposer_interface.py,True,7,proposer interface and candidate schema
1,tests/test_proposer_prompt.py,True,10,prompt structure and leakage audit
2,tests/test_proposer_stub.py,True,8,deterministic stub scenarios
3,tests/test_orchestrator_ingest.py,True,13,ingest lifecycle and attempt accounting
4,tests/test_budget_ledger.py,True,14,pre-charge ledger and crash safety


,evidence,reported
0,new D6 test files,5
1,new D6 tests/effective cases,58
2,full suite,"648 passing, 0 regressions"


D6 test evidence checks


,check,status,detail
0,all D6 test files exist,PASS,test_file exists ...
1,five D6 test files are represented,PASS,
2,"test themes cover interface, prompt, stub, ing...",PASS,
3,reported full-suite evidence is recorded,PASS,


## K. D6 Stage 1 Verdict

Acceptance checklist:

- proposer interface is frozen enough for Stage 2
- prompt leakage audit is in place
- deterministic stub covers major contract scenarios
- ingest classifies valid / invalid / duplicate / complexity / grammar failures deterministically
- invalid DSL counts as attempted
- duplicates do not become fresh hypotheses
- crash-safe pre-charge ledger works
- live-model integration is intentionally deferred

D6 Stage 1 passes acceptance and is ready for Stage 2 live Sonnet integration.

In [12]:
verdict_rows = [
    {"gate": "delivered components", "pass": D6_ACCEPTANCE.get("delivered_components", False)},
    {"gate": "frozen batch-start boundaries", "pass": D6_ACCEPTANCE.get("frozen_boundaries", False)},
    {"gate": "prompt builder + leakage audit", "pass": D6_ACCEPTANCE.get("prompt_leakage_audit", False)},
    {"gate": "deterministic stub scenario coverage", "pass": D6_ACCEPTANCE.get("stub_scenario_coverage", False)},
    {"gate": "ingest lifecycle classification", "pass": D6_ACCEPTANCE.get("ingest_lifecycle", False)},
    {"gate": "attempt accounting discipline", "pass": D6_ACCEPTANCE.get("attempt_accounting", False)},
    {"gate": "crash-safe budget ledger", "pass": D6_ACCEPTANCE.get("budget_ledger", False)},
    {"gate": "backend replaceability", "pass": D6_ACCEPTANCE.get("backend_replaceability", False)},
    {"gate": "test evidence recorded", "pass": D6_ACCEPTANCE.get("test_evidence", False)},
]
verdict_df = pd.DataFrame(verdict_rows)
verdict_df["status"] = verdict_df["pass"].map({True: "PASS", False: "FAIL"})
display(verdict_df[["gate", "status"]])

assert verdict_df["pass"].all(), verdict_df[~verdict_df["pass"]]

print("D6 Stage 1 acceptance verdict: PASS")
print("Stage 1 proves loop discipline without introducing model variability.")
print("Stage 2 must swap in the live Sonnet proposer backend without relaxing any Stage 1 accounting, leakage, or frozen-search-space constraints.")

,gate,status
0,delivered components,PASS
1,frozen batch-start boundaries,PASS
2,prompt builder + leakage audit,PASS
3,deterministic stub scenario coverage,PASS
4,ingest lifecycle classification,PASS
5,attempt accounting discipline,PASS
6,crash-safe budget ledger,PASS
7,backend replaceability,PASS
8,test evidence recorded,PASS


D6 Stage 1 acceptance verdict: PASS
Stage 1 proves loop discipline without introducing model variability.
Stage 2 must swap in the live Sonnet proposer backend without relaxing any Stage 1 accounting, leakage, or frozen-search-space constraints.
